# 3/3 — Matched evaluation for the no-person cls_pw sweep (no GPU needed)

Scores every model on exactly the same images, so the numbers are comparable:

- the three no-person runs (`cls_pw` 0.0 / 0.5 / 1.0): their own validation split (all 47 classes) and the catastrophic-forgetting set (the 19 kept COCO classes)
- run 2 (the deployed 48-class model, with `person`): the same forgetting set, with labels shifted into its 48-class scheme
- unmodified base `yolo26n.pt`: the same forgetting images with original COCO ids

The forgetting set is every COCO `val2017` image containing at least one kept class, built by `visionassist-01-dataprep-noperson`. Writes `matched_eval.json`.


In [ ]:
!echo "--- /kaggle/input contents ---"; ls -la /kaggle/input/ 2>&1; ls -la /kaggle/input/*/ 2>&1; ls -la /kaggle/input/*/*/ 2>&1


In [ ]:
!pip install -q ultralytics==8.4.106


In [ ]:
from pathlib import Path

# kernel_sources mount paths are not stable (see RUN_LOG.md finding 2), so search by
# file name + parent directory rather than hardcoding a path shape.
def _find_one(filename, parent_dir_name):
    matches = [p for p in Path("/kaggle/input").rglob(filename) if p.parent.name == parent_dir_name]
    assert len(matches) == 1, f"Expected exactly one {parent_dir_name}/{filename}, found {matches}"
    return matches[0]

TRAIN_VAL_YAML = _find_one("lostfound.yaml", "lostfound")
FORGETTING_YAML = _find_one("lostfound.yaml", "coco_forgetting_check")
COCO80_YAML = _find_one("coco80.yaml", "coco_forgetting_check_coco80")

# One best.pt per attached training kernel; tell them apart by the kernel slug in the path.
RUN_TAGS = {"noperson_clspw0.0": "clspw00", "noperson_clspw0.5": "clspw05", "noperson_clspw1.0": "clspw10", "run2_with_person_clspw0.5": "sample5000"}
all_best = [p for p in Path("/kaggle/input").rglob("best.pt") if p.parent.name == "weights"]
WEIGHTS = {}
for run, tag in RUN_TAGS.items():
    hits = [p for p in all_best if tag in str(p)]
    assert len(hits) == 1, f"Expected exactly one best.pt for {run} (path containing '{tag}'), found {hits} among {all_best}"
    WEIGHTS[run] = hits[0]
for k, v in {"TRAIN_VAL_YAML": TRAIN_VAL_YAML, "FORGETTING_YAML": FORGETTING_YAML, "COCO80_YAML": COCO80_YAML, **WEIGHTS}.items():
    print(f"{k}: {v}")


In [ ]:
import yaml, os

# Run 2 was trained on the 48-class scheme (person at index 0, everything else one
# higher). Build a copy of the forgetting set's labels shifted by +1 so it can be scored
# on the same images. Kept under /kaggle/tmp so it is not committed as Output.
names47 = yaml.safe_load(FORGETTING_YAML.read_text())["names"]
assert len(names47) == 47 and "person" not in names47, names47
fc_dir = FORGETTING_YAML.parent
fc48 = Path("/kaggle/tmp/fc48")
(fc48 / "labels" / "val").mkdir(parents=True, exist_ok=True)
if not (fc48 / "images").exists():
    os.symlink(fc_dir / "images", fc48 / "images")
n = 0
for lbl in (fc_dir / "labels" / "val").glob("*.txt"):
    out = []
    for line in lbl.read_text().splitlines():
        parts = line.split()
        if parts:
            out.append(f"{int(parts[0]) + 1} {' '.join(parts[1:])}")
    (fc48 / "labels" / "val" / lbl.name).write_text("\n".join(out) + "\n")
    n += 1
names48 = ["person"] + names47
(fc48 / "lostfound.yaml").write_text(f"train: images/val\nval: images/val\n\nnc: 48\nnames: {names48!r}\n")
FC48_YAML = fc48 / "lostfound.yaml"
print(f"Wrote {n} shifted label files to {fc48}")


In [ ]:
import json
from ultralytics import YOLO

def per_class(metrics):
    box = metrics.box
    out = {}
    for i, ci in enumerate(box.ap_class_index):
        out[metrics.names[int(ci)]] = {"ap50": float(box.ap50[i]), "ap50_95": float(box.ap[i]), "precision": float(box.p[i]), "recall": float(box.r[i])}
    return out

def evaluate(weights, data_yaml):
    m = YOLO(str(weights)).val(data=str(data_yaml), split="val", imgsz=640, plots=False, verbose=False)
    return {"map50": float(m.box.map50), "map50_95": float(m.box.map), "per_class": per_class(m), "speed_ms": dict(m.speed)}

results = {"forgetting_set_images": len(list((fc_dir / "images" / "val").glob("*"))), "runs": {}}

results["runs"]["base_yolo26n"] = {"forgetting": evaluate("yolo26n.pt", COCO80_YAML)}
json.dump(results, open("/kaggle/working/matched_eval.json", "w"), indent=2)

results["runs"]["run2_with_person_clspw0.5"] = {"forgetting": evaluate(WEIGHTS["run2_with_person_clspw0.5"], FC48_YAML)}
json.dump(results, open("/kaggle/working/matched_eval.json", "w"), indent=2)

for run in ("noperson_clspw0.0", "noperson_clspw0.5", "noperson_clspw1.0"):
    results["runs"][run] = {
        "forgetting": evaluate(WEIGHTS[run], FORGETTING_YAML),
        "own_val": evaluate(WEIGHTS[run], TRAIN_VAL_YAML),
    }
    json.dump(results, open("/kaggle/working/matched_eval.json", "w"), indent=2)
print("Wrote /kaggle/working/matched_eval.json")


In [ ]:
kept = names47[:19]
runs = list(results["runs"])
print("mAP50-95 on the forgetting set, 19 kept classes (same images for every model)")
print(f"{'class':14s} " + " ".join(f"{r[:22]:>24s}" for r in runs))
for name in kept:
    print(f"{name:14s} " + " ".join(f"{results['runs'][r]['forgetting']['per_class'].get(name, {}).get('ap50_95', float('nan')):24.3f}" for r in runs))
print(f"{'MEAN (19)':14s} " + " ".join(f"{sum(results['runs'][r]['forgetting']['per_class'].get(n, {}).get('ap50_95', 0.0) for n in kept) / len(kept):24.3f}" for r in runs))
print()
for r in runs:
    if "own_val" in results["runs"][r]:
        ov = results["runs"][r]["own_val"]
        novel = [ov["per_class"][n]["ap50_95"] for n in names47[19:] if n in ov["per_class"]]
        print(f"{r}: own val mAP50 {ov['map50']:.3f}, mAP50-95 {ov['map50_95']:.3f}, novel-class mean mAP50-95 {sum(novel) / len(novel):.3f} ({len(novel)} classes with val instances)")
